In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, f1_score
from preprocessing import get_features_and_target
from visualizer import plot_visualizer, plot_visualizer_classification
import plotly.graph_objects as go
from sklearn.model_selection import StratifiedKFold
from preprocessing import get_features_and_target_classification
from tabpfn import TabPFNClassifier, TabPFNRegressor
import matplotlib.pyplot as plt

In [2]:
import huggingface_hub
huggingface_hub.login()

# Getting Dataframe

In [ ]:
# Load the training and development datasets
train_df = pd.read_csv("data/train_data.csv")
dev_df = pd.read_csv("data/development_data.csv")
#sc = StandardScaler()

target_column_class = "Category" 
 

x_train, y_train  = get_features_and_target(train_df, target_column_class)
x_dev, y_dev = get_features_and_target(dev_df, target_column_class)

#x_train = sc.fit_transform(X=x_train)
#x_dev = sc.transform(x_dev)

In [4]:
x_train

,Pressure (PSI),Welding Time (ms),Angle (Deg),Force (N),Current (A),Thickness A (mm),Thickness B (mm)
0,35,200,0,6.82,1081.47,0.922,0.920
1,35,1500,0,52.25,2014.73,0.920,0.925
2,95,200,0,16.57,1321.93,0.912,0.924
3,95,200,0,41.42,1615.83,0.948,0.939
4,35,1500,0,63.82,1137.29,0.930,0.937
...,...,...,...,...,...,...,...
271,60,1200,0,98.29,3506.33,0.632,0.627
272,60,1200,0,98.04,4226.94,0.647,0.624
273,60,1200,0,98.07,3601.67,0.666,0.633
274,60,1200,0,97.09,4161.74,0.615,0.619


# Add Physical Columns Interfacial_Failure and Pullout_Failure

In [5]:
def compute_interfacial_failure(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     f_pull = 1 * (np.pi/4) * (4 * np.sqrt(t))**2 * (0.7 * 365) 
     return np.round(f_pull, 1) 

def compute_pullout_failure(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     x = df[['Thickness A (mm)', 'Thickness B (mm)']].sum(axis=1)
     # x can be approximated to metal sheet thickness. Change 2*t either to t to use the thinner 
     # metal sheet or 2*x to test if the sum of both metal sheets give beter results
     f_pull = np.pi * ((4 * np.sqrt(t)) + 2*t)*t*365 
     return np.round(f_pull, 1) 

In [6]:
x_dev

,Pressure (PSI),Welding Time (ms),Angle (Deg),Force (N),Current (A),Thickness A (mm),Thickness B (mm)
0,95,1500,0,124.19,1045.90,0.918,0.925
1,35,200,0,6.58,1171.55,0.931,0.922
2,35,200,0,6.61,916.16,0.936,0.927
3,35,200,15,27.19,1191.93,0.933,0.927
4,95,1500,15,133.53,1304.90,0.949,0.994
...,...,...,...,...,...,...,...
88,60,1200,0,96.79,4293.96,0.621,0.625
89,60,1200,0,97.29,4226.13,0.634,0.625
90,60,1200,0,97.29,4311.15,0.623,0.630
91,60,1200,0,97.12,4738.77,0.612,0.622


# Fit Model

In [7]:
# Target value

classifier = TabPFNClassifier() 

classifier.fit(x_train,y_train)

predictions_class_train = classifier.predict(x_train)
predictions_class_dev = classifier.predict(x_dev)



# Add Pullforce as Target

In [8]:
def compute_pullforces(x, y, predictions_class):
    pullforces = []

    for sample_id, pred in zip(y.index, predictions_class):
        row_x_df = x.loc[[sample_id]]  # 1-row DataFrame

        if pred == "Bad":
            value = compute_interfacial_failure(row_x_df).iloc[0]
        else:
            value = compute_pullout_failure(row_x_df).iloc[0]

        pullforces.append(value)

    return np.array(pullforces)


In [9]:
predictions_class_train = classifier.predict(x_train)
x_train["Physical Inference"] = compute_pullforces(x_train, y_train, predictions_class_train)


predictions_class_dev = classifier.predict(x_dev)
x_dev["Physical Inference"] = compute_pullforces(x_dev, y_dev, predictions_class_dev)


In [10]:
x_train.head()

,Pressure (PSI),Welding Time (ms),Angle (Deg),Force (N),Current (A),Thickness A (mm),Thickness B (mm),Physical Inference
0,35,200,0,6.82,1081.47,0.922,0.920,2953.9
1,35,1500,0,52.25,2014.73,0.920,0.925,5988.6
2,95,200,0,16.57,1321.93,0.912,0.924,2928.2
3,95,200,0,41.42,1615.83,0.948,0.939,3014.9
4,35,1500,0,63.82,1137.29,0.930,0.937,6097.2


In [11]:
print(predictions_class_dev)

['Good' 'Bad' 'Bad' 'Bad' 'Good' 'Bad' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good'
 'Good' 'Good']


In [12]:
y_train_Regressor = train_df.groupby("Sample ID")['PullTest (N)'].first()
y_dev_Regressor = dev_df.groupby("Sample ID")['PullTest (N)'].first()

In [13]:
y_train_Regressor.head(10)

Sample ID
1     2127.7
2     5346.4
4     2350.4
5     2174.8
7     3897.5
9     1932.5
10    4174.4
11    3816.0
12    1832.5
14    3970.1
Name: PullTest (N), dtype: float64

In [14]:
x_train

,Pressure (PSI),Welding Time (ms),Angle (Deg),Force (N),Current (A),Thickness A (mm),Thickness B (mm),Physical Inference
0,35,200,0,6.82,1081.47,0.922,0.920,2953.9
1,35,1500,0,52.25,2014.73,0.920,0.925,5988.6
2,95,200,0,16.57,1321.93,0.912,0.924,2928.2
3,95,200,0,41.42,1615.83,0.948,0.939,3014.9
4,35,1500,0,63.82,1137.29,0.930,0.937,6097.2
...,...,...,...,...,...,...,...,...
271,60,1200,0,98.29,3506.33,0.632,0.627,3178.8
272,60,1200,0,98.04,4226.94,0.647,0.624,3153.9
273,60,1200,0,98.07,3601.67,0.666,0.633,3228.9
274,60,1200,0,97.09,4161.74,0.615,0.619,3079.6


# Fit 2nd Model

In [15]:
regressor = TabPFNRegressor()
regressor.fit(x_train, y_train_Regressor)

# Predict on the test set
predictions_regression_dev = regressor.predict(x_dev)


In [16]:
y_dev

0     Good
1      Bad
2      Bad
3      Bad
4     Good
      ... 
88    Good
89    Good
90    Good
91    Good
92    Good
Name: Category, Length: 93, dtype: object

# Check Validation Data

In [17]:
# Category array (must be aligned with y_dev)
categories = dev_df.groupby("Sample ID")["Category"].first().values

plot_visualizer(
    true_vals=y_dev_Regressor,
    pred_vals=predictions_regression_dev,
    categories=categories,
    title=f"Validation Samples: True vs Prediction (TabPFN) by Category"
)

# Check Validation Loss and R2

In [18]:
# Calculate MAE and RMSE and R2
mae  = mean_absolute_error(y_dev_Regressor, predictions_regression_dev)
rmse = np.sqrt(mean_squared_error(y_dev_Regressor, predictions_regression_dev))
R2   = r2_score(y_dev_Regressor, predictions_regression_dev)


print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {R2:.2f}")

MAE:  133.13
RMSE: 217.83
R2: 0.63

MAE:  120.31
RMSE: 200.66
R2: 0.67


# Cross Validation

In [ ]:
cross_df = pd.read_csv("data/train_dev_data.csv")
target_column_regression = "PullTest (N)" 

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_predictions = []
fold_true = []
fold_predictions_regression = []
fold_true_regression = []

for fold, (train_index, val_index) in enumerate(skf.split(cross_df["Sample ID"], cross_df["Category"])):
    x_tr, y_tr = get_features_and_target(cross_df.iloc[train_index], target_column_class) 
    x_val, y_val = get_features_and_target(cross_df.iloc[val_index], target_column_class)
    y_train_regression, y_val_regression = get_features_and_target(cross_df.iloc[val_index], target_column_regression)

    classifier = TabPFNClassifier()
    classifier.fit(x_tr,y_tr)

    predictions_class_train = classifier.predict(x_tr)
    predictions_class_dev = classifier.predict(x_val)


    # Plot classification results for this fold
    plot_visualizer_classification(
        y_true=y_val,
        y_pred=predictions_class_dev,
        class_labels=["Bad", "Good"],
        title=f"Fold {fold+1} — Confusion Matrix"
    )

    pullforces = []

    for i in range(len(predictions_class_dev)):
        row_x_df = x_val.iloc[i:i+1]   

        if predictions_class_dev[i] == "Bad":
            value = compute_interfacial_failure(row_x_df).iloc[0]
        else:
            value = compute_pullout_failure(row_x_df).iloc[0]

        pullforces.append(value)       

    pullforces = np.array(pullforces)


    x_tr.loc[:, "Physical Inference"] = compute_pullforces(x_tr, y_tr, predictions_class_train) 
    x_val.loc[:, "Physical Inference"] = compute_pullforces(x_val, y_val, predictions_class_dev)

    regressor = TabPFNRegressor()
    regressor.fit(x_tr, y_train_regression)

    # Predict on the test set
    predictions_regression_dev = regressor.predict(x_val)

    mae  = mean_absolute_error(y_val_regression, predictions_regression_dev)
    rmse = np.sqrt(mean_squared_error(y_val_regression, predictions_regression_dev))
    R2   = r2_score(y_val_regression, predictions_regression_dev)

    # Categories
    categories= cross_df.iloc[val_index]["Category"].values

    plot_visualizer(
        true_vals=y_val_regression,
        pred_vals=pullforces,
        categories=categories,
        title=f"Fold {fold+1}: True vs Prediction (TabPFN) by Category"
    )

    print(f"\nFold {fold+1}")
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R²  :", R2)

    fold_predictions_regression.append(predictions_regression_dev)
    fold_true_regression.append(y_val_regression.values)

    fold_predictions.append(predictions_class_dev)
    fold_true.append(y_val.values)

ValueError: Found input variables with inconsistent numbers of samples: [74, 93]

In [ ]:
# Final Evaluation over all Folds
y_true_all = np.concatenate(fold_true)
y_pred_all = np.concatenate(fold_predictions)

# F1 Score
f1 = f1_score(y_true_all, y_pred_all, average="macro")
print("F1 Score:", f1)

# Final Evaluation over all Folds
y_true_all_regression = np.concatenate(fold_true_regression)
y_pred_all_regression = np.concatenate(fold_predictions_regression)

mae  = mean_absolute_error(y_true_all_regression, y_pred_all_regression)
rmse = np.sqrt(mean_squared_error(y_true_all_regression, y_pred_all_regression))
R2   = r2_score(y_true_all_regression, y_pred_all_regression)

print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {R2:.2f}")

F1 Score: 0.9494912624903042
MAE:  119.16
RMSE: 206.08
R²:   0.71
